# Capítulo 3 · Del recuento a la red

Una red que aprende la tabla de bigramas del capítulo 2 corrigiendo puntuaciones con descenso de gradiente. Python estándar y CPU: no hay que instalar nada. Ejecuta las celdas en orden.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio: el mismo programa, por partes, para ejecutarlo y experimentar.

## 1. Los datos

Los mismos nombres, la misma reserva y las mismas marcas que en el capítulo 2. `transiciones` convierte los nombres en parejas (contexto, salida). La última constante reúne las decisiones del experimento: pasos, tasa y freno.

In [1]:
"""Capítulo 3: una red que aprende a contar. Solo biblioteca estándar y CPU."""
import math
import random

ENTRENAMIENTO = ("ana", "anna", "alba", "alan")
RESERVA = ("lana", "bala")
INICIO, FIN = "^", "$"
CONTEXTOS = (INICIO, "a", "b", "l", "n")
SALIDAS = ("a", "b", "l", "n", FIN)
PASOS, TASA, FRENO = 1000, 5.0, 0.05


def transiciones(nombres):
    parejas = []
    for nombre in nombres:
        cadena = INICIO + nombre + FIN
        parejas.extend(zip(cadena, cadena[1:]))
    return parejas

In [2]:
entrenamiento = transiciones(ENTRENAMIENTO)
reserva = transiciones(RESERVA)
print(len(entrenamiento), "transiciones de entrenamiento y", len(reserva), "de reserva")
print("Las de ana:", entrenamiento[:4])

19 transiciones de entrenamiento y 10 de reserva
Las de ana: [('^', 'a'), ('a', 'n'), ('n', 'a'), ('a', '$')]


## 2. De puntuaciones a pérdida

`softmax` convierte una fila de puntuaciones en probabilidades: exponencial de cada una y división entre la suma. `perdida` es la NLL media del capítulo 2: recibe probabilidades, así que sirve tanto para la red como para las tablas contadas.

In [3]:
def softmax(puntuaciones):
    exponenciales = [math.exp(z) for z in puntuaciones]
    total = sum(exponenciales)
    return [e / total for e in exponenciales]


def probabilidades(tabla):
    return {c: softmax(tabla[c]) for c in CONTEXTOS}


def perdida(probs, parejas):
    costes = []
    for c, s in parejas:
        p = probs[c][SALIDAS.index(s)]
        if p == 0:
            return math.inf
        costes.append(-math.log(p))
    return sum(costes) / len(costes)

Comprueba los dos ejemplos del capítulo: la fila del ejercicio 2 y una fila a cero, que da el modelo uniforme.

In [4]:
print("softmax([0, 0, 1, 2, 2]) =", [round(p, 3) for p in softmax([0, 0, 1, 2, 2])])
print("softmax([0, 0, 0, 0, 0]) =", [round(p, 3) for p in softmax([0, 0, 0, 0, 0])])

softmax([0, 0, 1, 2, 2]) = [0.051, 0.051, 0.139, 0.379, 0.379]
softmax([0, 0, 0, 0, 0]) = [0.2, 0.2, 0.2, 0.2, 0.2]


## 3. Aprender

`correcciones` aplica la regla de la salida observada: suma la probabilidad a todas las salidas y resta 1 en la que apareció. `paso` actualiza cada puntuación con el freno: `(1 - tasa * freno) * z - tasa * corrección`. `entrenar` parte de todas las puntuaciones a cero, es decir, del modelo uniforme.

In [5]:
def correcciones(tabla, parejas):
    grad = {c: [0.0] * len(SALIDAS) for c in CONTEXTOS}
    for c, s in parejas:
        p = softmax(tabla[c])
        for j, salida in enumerate(SALIDAS):
            observada = 1.0 if salida == s else 0.0
            grad[c][j] += (p[j] - observada) / len(parejas)
    return grad


def paso(tabla, parejas, tasa, freno):
    grad = correcciones(tabla, parejas)
    for c in CONTEXTOS:
        tabla[c] = [(1 - tasa * freno) * z - tasa * g
                    for z, g in zip(tabla[c], grad[c])]


def entrenar(parejas, pasos=PASOS, tasa=TASA, freno=0.0):
    tabla = {c: [0.0] * len(SALIDAS) for c in CONTEXTOS}
    historial = [(0, perdida(probabilidades(tabla), parejas))]
    for n in range(1, pasos + 1):
        paso(tabla, parejas, tasa, freno)
        if n in (1, 10, 100, 1000):
            historial.append((n, perdida(probabilidades(tabla), parejas)))
    return tabla, historial

Sigue a mano el primer paso de la fila de `a` y compara con el capítulo: corrección dividida entre 19 y probabilidades tras un paso con tasa 5.

In [6]:
tabla = {c: [0.0] * len(SALIDAS) for c in CONTEXTOS}
print("Corrección de la fila de a:", [round(g, 3) for g in correcciones(tabla, entrenamiento)["a"]])
paso(tabla, entrenamiento, TASA, 0.0)
print("Puntuaciones tras un paso:", [round(z, 2) for z in tabla["a"]])
print("Probabilidades tras un paso:", [round(p, 2) for p in softmax(tabla["a"])])

Corrección de la fila de a: [0.084, 0.084, -0.021, -0.074, -0.074]
Puntuaciones tras un paso: [-0.42, -0.42, 0.11, 0.37, 0.37]
Probabilidades tras un paso: [0.12, 0.12, 0.21, 0.27, 0.27]


## 4. Comparar y generar

`por_recuento` reconstruye las tablas del capítulo 2 como referencia. `generar` sortea nombres a partir de cualquier tabla de probabilidades. `ejecutar` entrena las dos redes y muestra todos los resultados.

In [7]:
def por_recuento(parejas, alpha):
    probs = {}
    for c in CONTEXTOS:
        cuentas = [sum(1 for a, s in parejas if a == c and s == salida)
                   for salida in SALIDAS]
        total = sum(cuentas) + alpha * len(SALIDAS)
        probs[c] = [(n + alpha) / total for n in cuentas]
    return probs


def generar(probs, rng, max_pasos=20):
    c, letras = INICIO, []
    for _ in range(max_pasos):
        s = rng.choices(SALIDAS, weights=probs[c], k=1)[0]
        if s == FIN:
            return "".join(letras), True
        letras.append(s)
        c = s
    return "".join(letras), False

In [8]:
def ejecutar():
    entrenamiento = transiciones(ENTRENAMIENTO)
    reserva = transiciones(RESERVA)
    red, historial = entrenar(entrenamiento)
    red_freno, _ = entrenar(entrenamiento, freno=FRENO)
    modelos = {
        "Uniforme": {c: [1 / len(SALIDAS)] * len(SALIDAS) for c in CONTEXTOS},
        "Recuento sin suavizar": por_recuento(entrenamiento, 0),
        "Recuento suavizado": por_recuento(entrenamiento, 1),
        "Red sin freno": probabilidades(red),
        "Red con freno": probabilidades(red_freno),
    }
    print("Pérdida de entrenamiento:", historial)
    for nombre, probs in modelos.items():
        print(f"{nombre:22} entrenamiento {perdida(probs, entrenamiento):.6f}"
              f"  reserva {perdida(probs, reserva):.6f}")
    for nombre in ("Recuento sin suavizar", "Red sin freno",
                   "Recuento suavizado", "Red con freno"):
        fila = [round(p, 3) for p in modelos[nombre]["a"]]
        print(f"Fila de a, {nombre}: {fila}")
    rng = random.Random(7)
    for _ in range(10):
        texto, completo = generar(modelos["Red con freno"], rng)
        print(repr(texto), "fin" if completo else "truncado")
    return modelos, historial


In [9]:
modelos, historial = ejecutar()

Pérdida de entrenamiento: [(0, 1.6094379124341003), (1, 1.2917141766358304), (10, 0.85891603260276), (100, 0.7582590042945252), (1000, 0.7485585431242812)]
Uniforme               entrenamiento 1.609438  reserva 1.609438
Recuento sin suavizar  entrenamiento 0.747513  reserva inf
Recuento suavizado     entrenamiento 1.113961  reserva 1.409950
Red sin freno          entrenamiento 0.748559  reserva 2.355192
Red con freno          entrenamiento 1.086813  reserva 1.438037
Fila de a, Recuento sin suavizar: [0.0, 0.0, 0.25, 0.375, 0.375]
Fila de a, Red sin freno: [0.0, 0.0, 0.25, 0.375, 0.375]
Fila de a, Recuento suavizado: [0.077, 0.077, 0.231, 0.308, 0.308]
Fila de a, Red con freno: [0.084, 0.084, 0.22, 0.306, 0.306]
'ablanbananabbnall' fin
'an' fin
'a' fin
'abalnannblaallbanba' fin
'banl' fin
'la' fin
'annanannl' fin
'a' fin
'anbn' fin
'ana' fin


## 5. Experimenta con el freno

Cambia el valor de `freno` y vuelve a ejecutar la celda (ejercicio 10). Observa la fila de `a` y la pérdida de entrenamiento. Escribe antes qué esperas que ocurra.

In [10]:
freno = 0.01  # prueba también 0.05 y 0.1
red, _ = entrenar(entrenamiento, freno=freno)
probs = probabilidades(red)
print("Fila de a:", [round(p, 3) for p in probs["a"]])
print("Pérdida de entrenamiento:", round(perdida(probs, entrenamiento), 4))

Fila de a: [0.032, 0.032, 0.235, 0.35, 0.35]
Pérdida de entrenamiento: 0.8703


## Ejercicios

1. Con una fila de puntuaciones [0; 0; 0; 0; 0], calcula las cinco probabilidades. ¿A qué modelo del capítulo 2 corresponde?

2. Calcula softmax de [0; 0; 1; 2; 2] sin mirar la tabla del capítulo. Comprueba que las probabilidades suman uno.

3. Suma 3 a todas las puntuaciones del ejercicio anterior y vuelve a calcular softmax. ¿Qué observas? Intenta explicar por qué ocurre.

4. Con todas las puntuaciones a cero, escribe las cinco correcciones de la transición `a`→`n`. ¿Cuál es negativa y qué le pasará a su puntuación?

5. Repite el ejercicio anterior si la red ya daba 0,9 a `n` y 0,025 a cada una de las otras salidas. ¿Qué cambia?

6. Reconstruye la corrección total de la fila de `a` en el primer paso, antes y después de dividir entre 19.

7. Comprueba con la fila de `n` que la corrección se anula cuando las probabilidades coinciden con las proporciones contadas.

8. ¿Por qué la red sin freno nunca da probabilidad exactamente cero a empezar por `l`? ¿Qué consecuencia tiene para `lana`?

9. Con tasa 5 y freno 0,05, ¿qué valor tendrá tras un paso una puntuación de 2 cuya corrección sea cero? ¿Y tras muchos pasos, si la corrección siguiera siendo cero?

10. Ejecuta el programa con un freno de 0,01 y de 0,1. Describe qué ocurre con la fila de `a` y con la pérdida de entrenamiento. No uses la reserva para decidir cuál es mejor.

11. Compara la red con freno y la tabla suavizada en la tabla de resultados. ¿En qué se parecen? ¿Son el mismo modelo?

12. En el capítulo 4 la red mirará tres letras. ¿Qué piezas de este capítulo se conservarán y cuál cambiará?

## Referencias

Idea docente: primera lección de [makemore](https://www.youtube.com/watch?v=PaCmpygFfXo), dentro de [Zero to Hero](https://github.com/karpathy/nn-zero-to-hero), de Andrej Karpathy. Aquí se ha reescrito con Python estándar. Colección mínima, código y explicaciones propios.